In [ ]:
import pandas as pd
import pyarrow as pa

In [ ]:
%time data_df = pd.read_csv("../../datasets/taxi-trips/taxi_trips_small.csv", low_memory=False)

### Take a look at the data

In [ ]:
data_df.head(3).T

### See how much memory is being used (in total and per column)

In [ ]:
def bytes2mb(b): return (b / (1024 ** 2))

In [ ]:
mem_s = data_df.memory_usage(deep=True, index=False)

In [ ]:
mem_df = pd.DataFrame(round(mem_s / (1024 ** 2), 2), columns=['mem_in_megs'])

In [ ]:
print("Total memory usage:", round(sum(mem_s) / (1024 ** 3), 2), "GB")

#### Memory usage per column (in megabytes)

In [ ]:
mem_df.sort_values("mem_in_megs")

### Combine mem usage, data types, unique values and sample values

In [ ]:
mem_df\
    .join(pd.DataFrame(data_df.dtypes, columns=["dtypes"]))\
    .join(pd.DataFrame(data_df.nunique(), columns=["nuniques"]))\
    .join(data_df[:1].T).sort_values("mem_in_megs")

#### Notice that "Payment Type" is a string, perhaps it is just category (with very few unique strings?)

In [ ]:
bytes2mb(data_df['Payment Type'].memory_usage(deep=True))

In [ ]:
data_df['Payment Type'].value_counts()

In [ ]:
data_df['Payment Type'] = data_df['Payment Type'].astype('category')

In [ ]:
data_df['Payment Type'].value_counts()

In [ ]:
bytes2mb(data_df['Payment Type'].memory_usage(deep=True))

In [ ]:
print("Total memory usage:", round(sum(data_df.memory_usage(deep=True, index=False)) / (1024 ** 3), 2), "GB")

We went from almost 700 megs to only 10 megs!!!

#### Check out Trip Start/End Timestamp, looks like they are supposed to be timestamps, but pandas thinks they are strings

In [ ]:
bytes2mb(data_df['Trip Start Timestamp'].memory_usage(deep=True)) \
    + bytes2mb(data_df['Trip End Timestamp'].memory_usage(deep=True))

In [ ]:
data_df['Trip Start Timestamp'] = pd.to_datetime(data_df['Trip Start Timestamp'][:10], format="%m/%d/%Y %H:%M:%S %p")
data_df['Trip End Timestamp'] = pd.to_datetime(data_df['Trip End Timestamp'][:10], format="%m/%d/%Y %H:%M:%S %p")

In [ ]:
bytes2mb(data_df['Trip Start Timestamp'].memory_usage(deep=True)) \
    + bytes2mb(data_df['Trip End Timestamp'].memory_usage(deep=True))

In [ ]:
print("Total memory usage:", round(sum(data_df.memory_usage(deep=True, index=False)) / (1024 ** 3), 2), "GB")

#### Notice that Pickup/Dropoff Centroid Locations are strings made up of values already in lat/long, elsewhere in table...drop them!

In [ ]:
bytes2mb(data_df['Pickup Centroid Location'].memory_usage(deep=True)) \
    + bytes2mb(data_df['Dropoff Centroid  Location'].memory_usage(deep=True))

In [ ]:
data_df.drop(['Pickup Centroid Location', 'Dropoff Centroid  Location'], axis=1, inplace=True)

In [ ]:
print("Total memory usage:", round(sum(data_df.memory_usage(deep=True, index=False)) / (1024 ** 3), 2), "GB")

#### Can we convert other columns to categories and save lots of memory?

In [ ]:
bytes2mb(data_df['Company'].memory_usage(deep=True)) , \
bytes2mb(data_df['Company'].astype('category').memory_usage(deep=True))

In [ ]:
bytes2mb(data_df['Taxi ID'].memory_usage(deep=True)) , \
bytes2mb(data_df['Taxi ID'].astype('category').memory_usage(deep=True))

In [ ]:
bytes2mb(data_df['Trip ID'].memory_usage(deep=True)) , \
bytes2mb(data_df['Trip ID'].astype('category').memory_usage(deep=True))

In [ ]:
data_df['Company'] = data_df['Company'].astype('category')

In [ ]:
data_df['Taxi ID'] = data_df['Taxi ID'].astype('category')

#### Where do we stand?

In [ ]:
pd.DataFrame(bytes2mb(data_df.memory_usage(deep=True)), columns=["mem_in_megs"])\
    .join(pd.DataFrame(data_df.dtypes, columns=["dtypes"]))\
    .join(pd.DataFrame(data_df.nunique(), columns=["nuniques"]))\
    .join(data_df[:1].T).sort_values("mem_in_megs")

In [ ]:
print("Total memory usage:", round(sum(data_df.memory_usage(deep=True, index=False)) / (1024 ** 3), 2), "GB")

In [ ]:
%time data_df.to_feather("../../datasets/taxi-trips/taxi_trips_small_trimmed.feather")

In [ ]:
!ls -ltrhc ../../datasets/taxi-trips/taxi_trips*

### _Awesome_ reference:
https://www.dataquest.io/blog/pandas-big-data/